<a href="https://colab.research.google.com/github/SquareArachnid/CS176_EX5/blob/main/YOLO_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata
import subprocess
import os
import base64

token = userdata.get("GITHUB_TOKEN")

auth = base64.b64encode(
    f"x-access-token:{token}".encode()
).decode()

env = os.environ.copy()
env["GIT_CONFIG_COUNT"] = "1"
env["GIT_CONFIG_KEY_0"] = "http.extraheader"
env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {auth}"

try:
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/SquareArachnid/CS176_EX5.git",
            "/content/CS176_EX5"
        ],
        check=True,
        env=env
    )
finally:
    del token
    del auth
    del env

In [2]:
!pip install -q ultralytics gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 2.5 MB/s eta 0:00:00


In [3]:
from ultralytics import YOLO
from pathlib import Path

MODEL_PATH = Path(
    "/content/CS176_EX5/yolo26m/weights/best.pt"
)

model = YOLO(str(MODEL_PATH))

print("モデルの読み込みが完了しました！")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
モデルの読み込みが完了しました！


In [4]:
import gradio as gr
import cv2

def detect_pothole(image, confidence):

    if image is None:
        return None, "Please upload an image."

    results = model.predict(
        source=image,
        conf=confidence,
        verbose=False
    )

    result = results[0]

    annotated_image = cv2.cvtColor(
        result.plot(),
        cv2.COLOR_BGR2RGB
    )

    count = len(result.boxes)

    # No-detection handling
    if count == 0:
        message = (
            "No potholes detected. "
            "This does not necessarily mean the road is free of potholes."
        )
    else:
        message = f"Detected Potholes: {count}"

    return annotated_image, message


with gr.Blocks(title="Pothole Detection System") as demo:

    gr.Markdown("# 🛣️ Pothole Detection System")

    gr.Markdown(
        "This application uses a trained YOLO26m model "
        "to detect potholes in road images."
    )

    with gr.Row():

        with gr.Column():

            input_image = gr.Image(
                label="Upload Road Image",
                type="numpy"
            )

            confidence = gr.Slider(
                minimum=0.1,
                maximum=1.0,
                value=0.25,
                step=0.05,
                label="Confidence Threshold"
            )

            detect_button = gr.Button(
                "Detect Potholes",
                variant="primary"
            )

        with gr.Column():

            output_image = gr.Image(
                label="Detection Result"
            )

            output_text = gr.Textbox(
                label="Detection Information"
            )

    detect_button.click(
        fn=detect_pothole,
        inputs=[input_image, confidence],
        outputs=[output_image, output_text]
    )

    # User guidance
    gr.Markdown(
        """
        ### How to Interpret the Results

        - Bounding boxes indicate the locations of detected potholes.
        - Confidence scores indicate the model's confidence in each detection.
        - The detection count shows the number of potholes identified.

        ### Limitations

        - Detection accuracy may be affected by lighting, shadows,
          image quality, and viewing angles.
        - Small or partially hidden potholes may not be detected.
        - This application is intended to assist road inspection
          and should not replace professional assessment.
        """
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d4a33563e2ecad57ce.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
